In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import numpy as np
from dataclasses import dataclass

device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
print(device)

cuda


### Generalized dense Autoencoder on MNIST!

<img src="./images/thumbnail.png" alt="drawing" width="50%"/>

Actually works holy!

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

batch_size = 16

def read_idx3_images(path):
    with open(path, 'rb') as f:
        # skip the 16 byte header: magic number(4), count(4), rows(4), cols(4)
        data = np.fromfile(f, dtype=np.uint8, offset=16)
    # reshape to (#images, 784) 
    return data.reshape(-1, 784).astype(np.float32) / 255.0

train_images = read_idx3_images('data/t10k-images.idx3-ubyte')

x_train = torch.tensor(train_images, dtype=torch.float32, device=device)

train_ds = TensorDataset(x_train, x_train)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, )
print(x_train.shape)
print(f"1 epoch = {10000/batch_size} batches")


torch.Size([10000, 784])
1 epoch = 625.0 batches


In [3]:
# hyperparams
@dataclass
class autoencConfig:
    inputsize: int = 784
    bottleneck: int = 128
    lr: int = 9e-4

class AutoEncoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.encode = nn.Sequential(
                nn.Linear(784, 512),
                nn.ReLU(),
                nn.Linear(512, 256),
                nn.ReLU(),
                nn.Linear(256, 128),
                nn.ReLU()
        )
        
        self.bottleneck = nn.Linear(128, config.bottleneck)
        
        self.decode = nn.Sequential(
            nn.ReLU(),
            nn.Linear(config.bottleneck, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, config.inputsize)
        )
        
        self.ln = nn.LayerNorm(config.inputsize)
        
        
    def forward(self, x, targets=None):
        x = self.encode(x)
        x = self.bottleneck(x)
        x = self.decode(x)
        
        if targets is None:
            loss = None
        else:
            loss = nn.MSELoss()(torch.sigmoid(self.ln(x)), targets)
            # loss = F.binary_cross_entropy_with_logits(self.ln(x), targets)
        return x, loss

In [ ]:
model = AutoEncoder(autoencConfig())
model.load_state_dict(torch.load("pt/mnister.pt", weights_only=True))
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=model.config.lr)
outputs = []
expected = []

In [5]:
for epochs in range(5):
    for images, labels in train_loader:
        optimizer.zero_grad(set_to_none=True)
        
        with torch.autocast(device_type=device, dtype=torch.bfloat16):
            output, loss = model(images, labels)

        loss.backward()
        optimizer.step()
        outputs.append(output.to(torch.float32))
        expected.append(labels.to(torch.float32))
    print(loss.item())
    

0.08362362533807755
0.05653664842247963
0.041281528770923615
0.03358843922615051
0.030205000191926956


In [6]:
import matplotlib.pyplot as plt

# img = Image.fromarray((arr * 255).clip(0, 255).astype(np.uint8), mode="L")
# img.save("reconstructed.png")

from ipywidgets import interact, Layout, IntSlider
def show_image(i):
    batch = i//batch_size
    print(f"epoch {batch/625}, batch {batch}, total examples: {i}")
    predicted = outputs[batch][i % batch_size].cpu().detach().reshape(28, 28).numpy() # 16 is batch
    e = expected[batch][i % batch_size].cpu().detach().reshape(28, 28).numpy()
    
    plt.figure(figsize=(6, 3))
    plt.subplot(1, 2, 1)
    plt.imshow(predicted, cmap="gray")
    plt.subplot(1, 2, 2)
    plt.imshow(e, cmap="gray")
    plt.show()
    
interact(show_image, i=IntSlider(min=0, max=batch_size*(len(outputs)-1), layout=Layout(width='75%')))

interactive(children=(IntSlider(value=0, description='i', layout=Layout(width='75%'), max=49984), Output()), _…

<function __main__.show_image(i)>

In [7]:
torch.save(model.state_dict(), "mnister.pt")


<All keys matched successfully>